# Human-in-the-Loop Workflow with Microsoft Agent Framework

## 🎯 Learning Objectives

In this notebook, you'll learn how to implement **human-in-the-loop** workflows using Microsoft Agent Framework's `ctx.request_info()` pattern. This powerful pattern allows you to pause AI workflows to gather human input, making your agents interactive and giving humans control over critical decisions.

## 🔄 What is Human-in-the-Loop?

**Human-in-the-loop (HITL)** is a design pattern where AI agents pause execution to request human input before continuing. This is essential for:

- ✅ **Critical decisions** - Get human approval before taking important actions
- ✅ **Ambiguous situations** - Let humans clarify when AI is uncertain
- ✅ **User preferences** - Ask users to choose between multiple options
- ✅ **Compliance & safety** - Ensure human oversight for regulated operations
- ✅ **Interactive experiences** - Build conversational agents that respond to user input

## 🏗️ How It Works in Microsoft Agent Framework

The framework provides two key building blocks for HITL, both used from inside a custom `Executor`:

1. **`ctx.request_info(payload, response_type)`** - Called from a `@handler` method. Pauses the workflow and emits a `WorkflowEvent` with `type="request_info"`
2. **`@response_handler`** - A second method on the same executor that receives the correlated human answer once it's supplied, matched by the payload/response types

**Workflow Pattern:**
```
Agent detects need for input
    ↓
Executor calls ctx.request_info(payload, response_type)
    ↓
Workflow pauses & emits a WorkflowEvent(type="request_info")
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application calls workflow.run(responses={request_id: answer}, stream=True)
    ↓
The executor's @response_handler method runs with the human's answer
```

## 🏨 Our Example: Hotel Booking with User Confirmation

We'll build upon the conditional workflow by adding human confirmation **before** suggesting alternative destinations:

1. User requests a destination (e.g., "Paris")
2. `availability_agent` checks if rooms are available
3. **If no rooms** → `confirmation_agent` asks "Would you like to see alternatives?"
4. Workflow **pauses** using `ctx.request_info()` inside `decision_manager`
5. **Human responds** "yes" or "no" via console input
6. `decision_manager` routes based on response:
   - **Yes** → Show alternative destinations
   - **No** → Cancel booking request
7. Display final result

This demonstrates how to give users control over the agent's suggestions!

---

Let's get started! 🚀

## Step 1: Import Required Libraries

We import the standard Agent Framework components plus **human-in-the-loop specific names**:
- `WorkflowEvent` - Unified event class for everything the workflow emits (`event.type` tells you if it's `"request_info"`, `"output"`, `"status"`, ...)
- `WorkflowRunState` - Enum of workflow run states
- `handler` / `response_handler` - Decorators to expose executor methods; `response_handler` correlates a human/external answer back to the request that asked for it
- `tool` - Decorator for turning a plain function into an agent tool (replaces the older `ai_function` name)

In [2]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowEvent,              # NEW: unified event for all workflow emissions (event.type == "request_info" / "output" / "status" / ...)
    WorkflowRunState,           # NEW: Enum of workflow run states
    tool,                       # NEW: replaces the old `ai_function` decorator name
    executor,
    handler,                    # NEW: Decorator for executor methods
    response_handler,           # NEW: Decorator that correlates a human/external response with its original request
)

# 🤖 GitHub Models or OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop components loaded: ctx.request_info(), @response_handler, WorkflowEvent")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: ctx.request_info(), @response_handler, WorkflowEvent


## Step 2: Define Pydantic Models for Structured Outputs

These models define the **schema** that agents will return. We keep all models from the conditional workflow and add:

**New for Human-in-the-Loop:**
- `HumanFeedbackRequest` - Plain dataclass that defines the request payload passed to `ctx.request_info()`
  - Contains `prompt` (question to ask) and `destination` (context about the unavailable city)

In [3]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass carrying the payload passed to ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request payload sent via ctx.request_info() asking if user wants alternatives.

    In current Agent Framework versions, request payloads are plain dataclasses -
    there is no RequestInfoMessage base class to subclass anymore.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (request_info payload) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (request_info payload) 🆕


## Step 3: Create the Hotel Booking Tool

Same tool from the conditional workflow - checks if rooms are available in the destination.

In [4]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @tool decorator


## Step 4: Define Condition Functions for Routing

We need **four condition functions** for our human-in-the-loop workflow:

**From conditional workflow:**
1. `has_availability_condition` - Routes when hotels ARE available
2. `no_availability_condition` - Routes when hotels are NOT available

**New for human-in-the-loop:**
3. `user_wants_alternatives_condition` - Routes when user says "yes" to alternatives
4. `user_declines_alternatives_condition` - Routes when user says "no" to alternatives

In [5]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Step 5: Create the Decision Manager Executor

This is the **core of the human-in-the-loop pattern**! The `DecisionManager` is a custom `Executor` with two handlers:

1. **`ask_human`** (`@handler`) - Receives `confirmation_agent`'s question and calls `ctx.request_info(...)` to pause the workflow
2. **`on_human_feedback`** (`@response_handler`) - Receives the correlated human answer once it's provided, and routes the workflow

Key features:
- Uses `@handler` to expose `ask_human` as a normal workflow step
- Uses `@response_handler` so `on_human_feedback` is invoked automatically with `(original_request, response, ctx)` once a matching answer arrives
- Yields simple "yes" or "no" messages that trigger our condition functions

In [6]:
class DecisionManager(Executor):
    """
    Bridges confirmation_agent's question to the human, then routes the workflow
    based on the human's answer.

    Current Agent Framework versions have no separate RequestInfoExecutor node in the
    graph. Instead, an executor pauses the workflow by calling ctx.request_info(...)
    from a @handler method, and receives the correlated human answer in a
    @response_handler method on the SAME executor (matched by request/response type).
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def ask_human(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """
        Receives confirmation_agent's structured question and pauses the workflow,
        asking the human via ctx.request_info(). This replaces the old
        prepare_human_request executor + RequestInfoExecutor pair.
        """
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>⏸️  Pausing workflow:</strong> Requesting human input for {confirmation.destination}
            </div>
        """)
        )

        await ctx.request_info(
            HumanFeedbackRequest(prompt=confirmation.question, destination=confirmation.destination),
            str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        response: str,
        ctx: WorkflowContext[AgentExecutorRequest],
    ) -> None:
        """
        Process human feedback and let the workflow route based on conditions.

        - original_request: the HumanFeedbackRequest with context (matched automatically
          by type to the payload passed to ctx.request_info() above)
        - response: the user's string reply (e.g., "yes" or "no")

        This handler just displays feedback and sends a plain routing message.
        The actual routing is done by condition functions on the edges.
        """
        user_reply = (response or "").strip().lower()
        destination = original_request.destination

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → Will route to cancellation_agent
                </div>
            """)
            )
            # Create and send a message for the cancellation_agent
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created with @handler (asks the human) and @response_handler (routes on the answer)")

✅ DecisionManager executor created with @handler (asks the human) and @response_handler (routes on the answer)


## Step 6: Create Custom Display Executor

Same display executor from conditional workflow - yields final results as workflow output.

In [7]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_response.text)


print("✅ display_result executor created with @executor decorator")

✅ display_result executor created with @executor decorator


## Step 7: Load Environment Variables

Configure the LLM client (GitHub Models, Azure OpenAI, or OpenAI).

In [8]:
# Load environment variables
load_dotenv()

from azure.identity import AzureCliCredential

# Azure OpenAI via the Responses API. Sign in with `az login` for keyless Entra ID auth.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API,
# so this sample calls Azure OpenAI directly. OpenAIChatClient uses the Responses API.
chat_client = OpenAIChatClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    credential=AzureCliCredential(),
    model=os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o-mini"),
)

print("✅ Chat client configured with Azure OpenAI (Responses API)")


✅ Chat client configured with Azure OpenAI (Responses API)


## Step 8: Create AI Agents and Executors

We create **six workflow components**:

**Agents (wrapped in AgentExecutor):**
1. **availability_agent** - Checks hotel availability using the tool
2. **confirmation_agent** - 🆕 Prepares the human confirmation question
3. **alternative_agent** - Suggests alternative cities (when user says yes)
4. **booking_agent** - Encourages booking (when rooms available)
5. **cancellation_agent** - 🆕 Handles cancellation message (when user says no)

**Special Executor:**
6. **decision_manager** - 🆕 Custom executor that pauses for human input via `ctx.request_info()` and routes based on the response (already defined above)

In [9]:
def strip_tool_internals(messages: list[Message]) -> list[Message]:
    """Drop server-issued function_call/function_result/reasoning content before handing the
    conversation off to a different agent (and therefore a different session/response chain).

    gpt-5-mini (a reasoning model) pairs every function_call with a reasoning item that only the
    *same* service-side conversation is allowed to replay. Passing that content into a different
    agent's request makes the API reject the request with "function_call ... provided without its
    required reasoning item". Downstream agents only need the plain-text outcome, not the tool
    call trace, so we filter it out.
    """
    filtered: list[Message] = []
    for m in messages:
        contents = [c for c in m.contents if c.type not in ("function_call", "function_result", "text_reasoning")]
        if contents:
            filtered.append(Message(role=m.role, contents=contents))
    return filtered


# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
# context_filter strips availability_agent's tool-call trace (see strip_tool_internals above) -
# required because this agent receives availability_agent's AgentExecutorResponse directly.
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Use Pydantic model for agent output
    ),
    id="confirmation_agent",
    context_mode="custom",
    context_filter=strip_tool_internals,
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
# Also receives availability_agent's AgentExecutorResponse directly - same context_filter needed.
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
    context_mode="custom",
    context_filter=strip_tool_internals,
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# NEW: DecisionManager instance - asks the human (via ctx.request_info) and routes based on the answer
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>decision_manager</strong> 🆕 - Pauses for human input (ctx.request_info) and routes based on the response</li>
        </ul>
    </div>
""")
)


## Step 9: Build the Workflow with Human-in-the-Loop

Now we construct the workflow graph with **conditional routing** including the human-in-the-loop path:

**Workflow Structure:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
decision_manager             display_result
  (ctx.request_info PAUSE)
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Key Edges:**
- `availability_agent → confirmation_agent` (when no rooms)
- `confirmation_agent → decision_manager` (asks the human via `ctx.request_info()`, pausing the workflow)
- `decision_manager → alternative_agent` (when user says "yes")
- `decision_manager → cancellation_agent` (when user says "no")
- `availability_agent → booking_agent` (when rooms available)
- All paths end at `display_result`

In [10]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info()
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → decision_manager → <strong>PAUSE FOR HUMAN</strong><br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

C:\Users\lujan\AppData\Local\Temp\ipykernel_17372\4283413795.py:3: DeprecationWarning: `output_executors` is deprecated and will be removed in a future version; use `output_from` instead.
  WorkflowBuilder(


## Step 10: Run Test Case 1 - City WITHOUT Availability (Paris with Human Confirmation)

This test demonstrates the **full human-in-the-loop cycle**:

1. Request hotel in Paris
2. availability_agent checks → No rooms
3. confirmation_agent creates human-facing question
4. decision_manager calls `ctx.request_info()`, **pausing the workflow**
5. **Application detects the `request_info` event and prompts user in console**
6. User types "yes" or "no"
7. Application resumes the workflow via `workflow.run(responses=..., stream=True)`
8. decision_manager's `@response_handler` routes based on the response
9. Final result displayed

**Key Pattern:**
- Use `workflow.run(message, stream=True)` for the first iteration
- Use `workflow.run(responses=pending_responses, stream=True)` for subsequent iterations
- Listen for `event.type == "request_info"` to detect when human input is needed
- Listen for `event.type == "output"` to capture final results

In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → decision_manager → <strong>PAUSE</strong> → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern
pending_responses: dict[str, str] | None = None
completed = False
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

while not completed:
    # First iteration streams the initial request.
    # Subsequent iterations stream the collected human responses back in via
    # workflow.run(responses=..., stream=True) - there is no separate
    # send_responses_streaming() method anymore, run() handles both cases.
    if pending_responses:
        print(f"\n📤 Sending human responses: {pending_responses}")
        stream = workflow.run(responses=pending_responses, stream=True)
        pending_responses = None  # Clear immediately after sending
    else:
        print(f"\n🚀 Starting workflow with request: 'I want to book a hotel in Paris'")
        stream = workflow.run(request_paris, stream=True)
    
    # Collect all events from this iteration
    events = [event async for event in stream]
    
    # Process events
    requests: list[tuple[str, str]] = []  # (request_id, prompt)
    
    for event in events:
        # Check for human input requests (WorkflowEvent with type == "request_info")
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            requests.append((event.request_id, event.data.prompt))
        
        # Check for workflow outputs (WorkflowEvent with type == "output")
        elif event.type == "output":
            workflow_output = str(event.data)
            completed = True
            print(f"\n✅ Workflow completed with output!")
    
    # If we have human requests, prompt the user
    if requests and not completed:
        responses: dict[str, str] = {}
        for req_id, prompt in requests:
            print(f"\n{'='*60}")
            print(f"💬 QUESTION FOR YOU:")
            print(f"   {prompt}")
            print(f"{'='*60}")
            
            # Get user input (in notebook, this will pause execution)
            answer = input("👉 Enter 'yes' or 'no': ").strip().lower()
            
            print(f"\n📝 You answered: {answer}")
            responses[req_id] = answer
        
        pending_responses = responses

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: d2f821df-4341-4d94-b115-b904ff1e1b74
   Destination: Paris

💬 QUESTION FOR YOU:
   Would you like me to show nearby alternative destinations?


## Step 11: Run Test Case 2 - City WITH Availability (Stockholm - No Human Input Needed)

This test demonstrates the **direct path** when rooms are available:

1. Request hotel in Stockholm
2. availability_agent checks → Rooms available ✅
3. booking_agent suggests booking
4. display_result shows confirmation
5. **No human input required!**

The workflow bypasses the human-in-the-loop path entirely when rooms are available.

In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Key Takeaways and Human-in-the-Loop Best Practices

### ✅ What You've Learned:

#### 1. **The `ctx.request_info()` / `@response_handler` Pattern**
The human-in-the-loop pattern in Microsoft Agent Framework uses two pieces on a custom `Executor`:
- `ctx.request_info(payload, response_type)` - Pauses the workflow and emits a `request_info` event
- `@response_handler` - A method on the same executor that receives the correlated answer once supplied

**Critical Understanding:**
- `ctx.request_info()` does NOT collect input itself - it only pauses the workflow and emits an event
- Your application code must listen for `WorkflowEvent` objects with `type == "request_info"` and collect input
- You must resume with `workflow.run(responses={request_id: answer}, stream=True)`

#### 2. **Streaming Execution Pattern**
```python
# First iteration
stream = workflow.run(initial_request, stream=True)

# Subsequent iterations (after human input)
stream = workflow.run(responses=pending_responses, stream=True)

# Always process events
events = [event async for event in stream]
```

#### 3. **Event-Driven Architecture**
Every workflow emission is a single `WorkflowEvent` class, discriminated by `event.type`:
- `"request_info"` - Human input is needed (workflow paused); use `event.request_id` and `event.data`
- `"output"` - Final result is available (workflow complete); use `event.data`
- `"status"` - State changes (`event.state`, e.g. `IN_PROGRESS`, `IDLE_WITH_PENDING_REQUESTS`, etc.)

#### 4. **Custom Executors with @handler and @response_handler**
The `DecisionManager` demonstrates how to create executors that:
- Use `@handler` to expose a method as a normal workflow step (`ask_human`)
- Use `@response_handler` to receive a typed `(original_request, response, ctx)` once a human answer arrives
- Route workflow by sending messages to other executors
- Access context via `WorkflowContext`

#### 5. **Conditional Routing with Human Decisions**
You can create condition functions that evaluate human responses:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_response.text.lower()
    return response_text == "yes"
```

### 🎯 Real-World Applications:

1. **Approval Workflows**
   - Get manager approval before processing expense reports
   - Require human review before sending automated emails
   - Confirm high-value transactions before execution

2. **Content Moderation**
   - Flag questionable content for human review
   - Ask moderators to make final decision on edge cases
   - Escalate to humans when AI confidence is low

3. **Customer Service**
   - Let AI handle routine questions automatically
   - Escalate complex issues to human agents
   - Ask customer if they want to speak to a human

4. **Data Processing**
   - Ask humans to resolve ambiguous data entries
   - Confirm AI interpretations of unclear documents
   - Let users choose between multiple valid interpretations

5. **Safety-Critical Systems**
   - Require human confirmation before irreversible actions
   - Get approval before accessing sensitive data
   - Confirm decisions in regulated industries (healthcare, finance)

6. **Interactive Agents**
   - Build conversational bots that ask follow-up questions
   - Create wizards that guide users through complex processes
   - Design agents that collaborate with humans step-by-step

### 🔄 Comparison: With vs Without Human-in-the-Loop

| Feature | Conditional Workflow | Human-in-the-Loop Workflow |
|---------|---------------------|---------------------------|
| **Execution** | Single `await workflow.run()` | Loop with `workflow.run(..., stream=True)` (both first call and resumes) |
| **User Input** | None (fully automated) | Interactive prompts via `input()` or UI |
| **Components** | Agents + Executors | + an executor that calls `ctx.request_info()` / `@response_handler` |
| **Events** | AgentExecutorResponse only | `WorkflowEvent` with `type in {"request_info", "output", ...}` |
| **Pausing** | No pausing | Workflow pauses when `ctx.request_info()` is awaited |
| **Human Control** | No human control | Humans make key decisions |
| **Use Case** | Automation | Collaboration & oversight |

### 🚀 Advanced Patterns:

#### Multiple Human Decision Points
You can have multiple executors calling `ctx.request_info()` in the same workflow - each just needs its own `@response_handler`:
```python
.add_edge(agent1, decision_point_1)  # calls ctx.request_info() -> first human decision
.add_edge(decision_point_1, agent2, condition=...)
.add_edge(agent2, decision_point_2)  # calls ctx.request_info() -> second human decision
.add_edge(decision_point_2, final_agent, condition=...)
```

#### Timeout Handling
Implement timeouts for human responses:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Default to safe option
```

#### Rich UI Integration
Instead of `input()`, integrate with web UI, Slack, Teams, etc.:
```python
if event.type == "request_info":
    # Send notification to user's preferred channel
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Conditional Human-in-the-Loop
Only ask for human input in specific situations:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Only route to human if confidence is low or value is high
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Best Practices:

1. **Keep Request Payloads Explicit**
   - Use a small dataclass for each request type (e.g. `HumanFeedbackRequest`)
   - Provides clarity and type safety for `ctx.request_info(payload, response_type)`
   - Enables rich context for UI rendering

2. **Use Descriptive Prompts**
   - Include context about what you're asking
   - Explain consequences of each choice
   - Keep questions simple and clear

3. **Handle Unexpected Input**
   - Validate user responses
   - Provide defaults for invalid input
   - Give clear error messages

4. **Track Request IDs**
   - Use the correlation between `request_id` and responses
   - Don't try to manage state manually

5. **Design for Non-Blocking**
   - Don't block threads waiting for input
   - Use async patterns throughout
   - Support concurrent workflow instances

### 📚 Related Concepts:

- **Agent Middleware** - Intercept agent calls (previous notebook)
- **Workflow State Management** - Persist workflow state between runs
- **Multi-Agent Collaboration** - Combine human-in-the-loop with agent teams
- **Event-Driven Architectures** - Build reactive systems with events

---

### 🎓 Congratulations!

You've mastered human-in-the-loop workflows with Microsoft Agent Framework! You now know how to:
- ✅ Pause workflows to gather human input
- ✅ Use `ctx.request_info()` and `@response_handler`
- ✅ Handle streaming execution with `WorkflowEvent`
- ✅ Create custom executors with `@handler`
- ✅ Route workflows based on human decisions
- ✅ Build interactive AI agents that collaborate with humans

**This is a critical pattern for building trustworthy, controllable AI systems!** 🚀